# 04 — WMAP9 + DESI DR1 Joint ΛCDM Fitting

**Author:** Cherian Parangot Ittyipe (FRAS 41871)
**Thesis DOI:** https://doi.org/10.5281/zenodo.17681763

---

## Joint likelihood

ln L_total(θ) = ln L_WMAP9(θ) + ln L_DESI_BAO(θ)

θ = [Ωbh², Ωch², H₀, nₛ, ln(10¹⁰Aₛ), τ]

**WMAP9** constrains all 6 parameters via the CMB TT/TE/EE/BB power spectra.
**DESI DR1 BAO** constrains Ωbh², Ωch², H₀ via distance ratios DM/rd and DH/rd.
nₛ, Aₛ, τ are insensitive to BAO and remain determined by WMAP9 alone.

## Key rule
No Planck rᵈ prior is used. WMAP9 already calibrates rᵈ through the CMB
acoustic peaks. Adding a Planck rᵈ prior would contaminate the joint result
with Planck information.

## Why the datasets are complementary
WMAP9 has degeneracies between H₀ and Ωch² along the acoustic scale direction.
DESI BAO breaks this degeneracy by measuring H₀rᵈ and Ωm at low redshift.
The joint posterior is tighter than either dataset alone on H₀ and Ωm.

## Prior bounds — widened from Notebook 02
The WMAP9-only notebook used tight bounds [60, 80] on H₀.
Here we widen to [55, 85] to allow the joint posterior to move freely.

## Cell 1 — Imports and Paths

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, ctypes, warnings
warnings.filterwarnings('ignore')
import camb
import emcee
import corner
import scipy.optimize as opt
from pathlib import Path
from datetime import datetime

BASE      = '/Users/cherianpi/Desktop/WMAP+DESI'
LIKE_DIR  = BASE + '/wmap_likelihood_v5'
PY_DIR    = BASE + '/Python files'
SO_PATH   = PY_DIR + '/libwmap9.so'
BAO_DIR   = BASE + '/bao_data'
CHAIN_DIR = PY_DIR + '/chains'
IMG_DIR   = PY_DIR + '/Images'
os.makedirs(CHAIN_DIR, exist_ok=True)
os.makedirs(IMG_DIR,   exist_ok=True)

PARAM_NAMES  = ['ombh2', 'omch2', 'H0', 'ns', 'ln10As', 'tau']
PARAM_LABELS = [r'$\Omega_b h^2$', r'$\Omega_c h^2$', r'$H_0$',
                r'$n_s$', r'$\ln(10^{10}A_s)$', r'$\tau$']

# Widened prior bounds relative to Notebook 02
# H0 extended to [55, 85] so joint posterior can move freely
PRIOR_BOUNDS = np.array([
    [0.018, 0.028],   # ombh2
    [0.08,  0.16 ],   # omch2
    [55.,   85.  ],   # H0   — widened from [60, 80]
    [0.9,   1.05 ],   # ns
    [2.9,   3.3  ],   # ln10As
    [0.04,  0.20 ],   # tau
])

NWALKERS = 32
NSTEPS   = 3000
NBURN    = 500

print('Paths:')
for k, v in [('LIKE_DIR', LIKE_DIR), ('SO_PATH', SO_PATH), ('BAO_DIR', BAO_DIR)]:
    status = 'OK' if os.path.exists(v) else 'NOT FOUND'
    print(f'  {k:10s}: [{status}]  {v}')

## Cell 2 — Load WMAP9 Likelihood Library

Exact setup from Notebook 02.

In [ ]:
lib = ctypes.CDLL(SO_PATH)
DP  = ctypes.c_double
DPA = ctypes.POINTER(DP)

INIT_SYM    = '__wmap_likelihood_9yr_MOD_wmap_likelihood_init'
COMPUTE_SYM = '__wmap_likelihood_9yr_MOD_wmap_likelihood_compute'

fn_init = getattr(lib, INIT_SYM)
fn_init.restype  = None
fn_init.argtypes = []

fn_compute = getattr(lib, COMPUTE_SYM)
fn_compute.restype  = None
fn_compute.argtypes = [DPA, DPA, DPA, DPA, DPA]

_orig_dir = os.getcwd()
os.chdir(LIKE_DIR)
fn_init()
os.chdir(_orig_dir)

print('WMAP9 likelihood library loaded and initialised.')

NUM_WMAP   = 8
COMP_NAMES = [
    'TT high-l MASTER', 'TT low-l Gibbs', 'TT low-l det', 'Beam+ptsrc',
    'TE high-l chi2',   'TE high-l det',  'lowl TT/TE/EE/BB chi2', 'lowl TT/TE/EE/BB det',
]

def wmap_like(cltt, clte, clee, clbb):
    """
    Official WMAP9 likelihood.
    Inputs : Dl arrays shape (1199,), ell=2..1200, muK^2.
    Returns: (total -2lnL, component vector shape (8,))
    """
    def a(x): return np.ascontiguousarray(x, dtype=np.float64)
    lk = np.zeros(NUM_WMAP, dtype=np.float64)
    fn_compute(
        a(cltt).ctypes.data_as(DPA), a(clte).ctypes.data_as(DPA),
        a(clee).ctypes.data_as(DPA), a(clbb).ctypes.data_as(DPA),
        lk.ctypes.data_as(DPA)
    )
    return 2.0 * lk.sum(), 2.0 * lk

# Quick verification
tc = np.loadtxt(LIKE_DIR + '/data/test_cls_v5.dat')
lt, _ = wmap_like(tc[:,1], tc[:,4], tc[:,2], tc[:,3])
print(f'Verification -2lnL = {lt:.4f}  (expect 7557.97)')

## Cell 3 — CAMB Theory Engine (WMAP9 side)

Returns Dl arrays for ell=2..1200 as required by the WMAP9 Fortran library.

In [ ]:
ELL_WMAP = np.arange(2, 1201)   # length 1199

def get_cls_camb_wmap(ombh2, omch2, H0, ns, ln10As, tau):
    """
    Compute lensed CMB Dl spectra for WMAP9 likelihood.
    Returns TT, TE, EE, BB each shape (1199,) for ell=2..1200 in muK^2.
    """
    cp = camb.CAMBparams()
    cp.set_cosmology(H0=H0, ombh2=ombh2, omch2=omch2, tau=tau)
    cp.InitPower.set_params(ns=ns, As=np.exp(ln10As) * 1e-10)
    cp.set_for_lmax(1250, lens_potential_accuracy=0)
    res = camb.get_results(cp)
    Dl  = res.get_cmb_power_spectra(cp, CMB_unit='muK', raw_cl=False)['total']
    return Dl[ELL_WMAP, 0], Dl[ELL_WMAP, 3], Dl[ELL_WMAP, 1], Dl[ELL_WMAP, 2]

# Sanity check
p_test = [0.02264, 0.1138, 70.0, 0.972, 3.089, 0.089]
neg2, _ = wmap_like(*get_cls_camb_wmap(*p_test))
print(f'CAMB test -2lnL = {neg2:.4f}  (official test: 7557.97)')

## Cell 4 — Load DESI DR1 BAO Data

File stems verified from disk. See Notebook 03.1 for details.

In [ ]:
def load_mean(filepath):
    values = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            try:
                values.append(float(parts[1]))
            except (IndexError, ValueError):
                continue
    return np.array(values)

def load_cov(filepath):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            try:
                rows.append([float(x) for x in line.split()])
            except ValueError:
                continue
    return np.array(rows)

# DR1 tracer definitions — stems verified from disk
TRACERS = {
    'BGS':       {'stem': 'desi_2024_gaussian_bao_BGS_BRIGHT-21.5_GCcomb_z0.1-0.4', 'mode': 'iso',   'zeff': 0.295},
    'LRG1':      {'stem': 'desi_2024_gaussian_bao_LRG_GCcomb_z0.4-0.6',              'mode': 'aniso', 'zeff': 0.510},
    'LRG2':      {'stem': 'desi_2024_gaussian_bao_LRG_GCcomb_z0.6-0.8',              'mode': 'aniso', 'zeff': 0.706},
    'LRG3+ELG1': {'stem': 'desi_2024_gaussian_bao_LRG+ELG_LOPnotqso_GCcomb_z0.8-1.1','mode': 'aniso', 'zeff': 0.930},
    'ELG2':      {'stem': 'desi_2024_gaussian_bao_ELG_LOPnotqso_GCcomb_z1.1-1.6',   'mode': 'aniso', 'zeff': 1.317},
    'QSO':       {'stem': 'desi_2024_gaussian_bao_QSO_GCcomb_z0.8-2.1',              'mode': 'iso',   'zeff': 1.491},
    'Lya':       {'stem': 'desi_2024_gaussian_bao_Lya_GCcomb',                        'mode': 'aniso', 'zeff': 2.330},
}

bao_data = {}
n_total  = 0
for name, info in TRACERS.items():
    mean_file = Path(BAO_DIR) / f"{info['stem']}_mean.txt"
    cov_file  = Path(BAO_DIR) / f"{info['stem']}_cov.txt"
    if not mean_file.exists() or not cov_file.exists():
        print(f'WARNING: {name} files not found')
        continue
    mean_vec = load_mean(mean_file)
    cov_mat  = load_cov(cov_file)
    if mean_vec.size == 0 or cov_mat.size == 0:
        print(f'WARNING: {name} empty')
        continue
    bao_data[name] = {
        'mean': mean_vec,
        'icov': np.linalg.inv(cov_mat),
        'mode': info['mode'],
        'zeff': info['zeff'],
    }
    n_total += mean_vec.size
    print(f'Loaded {name:12s} | zeff={info["zeff"]:.3f} | mode={info["mode"]:5s} | ndata={mean_vec.size}')

print(f'\nLoaded {len(bao_data)}/{len(TRACERS)} tracers | Total measurements: {n_total}')

In [ ]:
# ===========================================================
#  RNAAS audit cell (v3) — fully self-contained.
#  Defines its own REQUIRED_KEYS and labelled loader, so it runs
#  in the OLD notebook that only has load_mean().
#  Requires: TRACERS, BAO_DIR, load_cov, get_bao_theory
# ===========================================================

import numpy as np
from pathlib import Path

FID = dict(ombh2=0.02239, omch2=0.11712, H0=67.41303)

REQUIRED_KEYS = {
    'aniso': ('DM_over_rs', 'DH_over_rs'),
    'iso':   ('DV_over_rs',),
}


def _rows(filepath):
    """(value, label) pairs in file order."""
    out = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            try:
                out.append((float(parts[1]), parts[2]))
            except ValueError:
                continue
    return out


def mean_labelled(filepath, mode):
    """Correct: assign by the label column."""
    d = {lab: val for val, lab in _rows(filepath)}
    return np.array([d[k] for k in REQUIRED_KEYS[mode]])


def mean_positional(filepath, mode):
    """Buggy: assign by row order, ignoring labels. This is load_mean()."""
    vals = [val for val, _ in _rows(filepath)]
    return np.array(vals[:len(REQUIRED_KEYS[mode])])


print(f'{"Tracer":<12}{"zeff":>7}{"chi2 positional":>18}{"chi2 labelled":>16}{"ratio":>12}')
print('-' * 65)

tot_pos = tot_lab = 0.0
rows_out = []

for name, info in TRACERS.items():
    mf = Path(BAO_DIR) / f"{info['stem']}_mean.txt"
    cf = Path(BAO_DIR) / f"{info['stem']}_cov.txt"
    if not (mf.exists() and cf.exists()):
        print(f'{name:<12} SKIPPED — files missing')
        continue

    mode = info['mode']
    try:
        icov = np.linalg.inv(load_cov(cf))
        theory, _ = get_bao_theory(FID['ombh2'], FID['omch2'], FID['H0'],
                                   info['zeff'], mode)
        theory = np.asarray(theory, dtype=float)

        d_lab = mean_labelled(mf, mode) - theory
        d_pos = mean_positional(mf, mode) - theory

        c_lab = float(d_lab @ icov @ d_lab)
        c_pos = float(d_pos @ icov @ d_pos)
    except Exception as e:
        print(f'{name:<12} FAILED — {type(e).__name__}: {e}')
        continue

    tot_lab += c_lab
    tot_pos += c_pos
    ratio = c_pos / c_lab if c_lab > 0 else float('nan')
    rows_out.append((name, info['zeff'], c_pos, c_lab, ratio))
    print(f'{name:<12}{info["zeff"]:7.3f}{c_pos:18.4f}{c_lab:16.4f}{ratio:12.2f}')

print('-' * 65)

if rows_out:
    tr = tot_pos / tot_lab if tot_lab > 0 else float('nan')
    print(f'{"TOTAL":<12}{"":>7}{tot_pos:18.4f}{tot_lab:16.4f}{tr:12.2f}')
    print()
    for r in rows_out:
        if r[0] == 'Lya':
            print(f'Lya alone  : {r[3]:.4f} -> {r[2]:.4f}   (factor {r[4]:.1f})')
    print(f'DESI total : {tot_lab:.4f} -> {tot_pos:.4f}   (factor {tr:.1f})')

In [ ]:
# ===========================================================
#  RNAAS audit cell — paste into 04_wmap9_desi_joint_lcdm.ipynb
#  AFTER the cell that defines get_bao_theory().
#
#  Produces:
#    (a) the literal row order of every mean file  -> the note's claim
#    (b) per-tracer chi^2 under a positional loader vs a labelled one
#    (c) the inflation ratio, defined unambiguously
# ===========================================================

import numpy as np
from pathlib import Path

# Fiducial point: joint posterior medians from run 20260628_004948
FID = dict(ombh2=0.02239, omch2=0.11712, H0=67.41303)


def load_mean_positional(filepath, mode):
    """The buggy loader: assign by ROW ORDER, ignoring the label column."""
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            try:
                rows.append(float(parts[1]))
            except ValueError:
                continue
    keys = REQUIRED_KEYS[mode]
    return dict(zip(keys, rows[:len(keys)]))


def raw_row_labels(filepath):
    """Return the label column in file order — this is the evidence."""
    labs = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) >= 3:
                labs.append(parts[2])
    return labs


# ---- (a) row order audit -----------------------------------
print('ROW ORDER AS STORED IN EACH _mean.txt FILE')
print('-' * 58)
for name, info in TRACERS.items():
    mf = Path(BAO_DIR) / f"{info['stem']}_mean.txt"
    if mf.exists():
        print(f'{name:12s} {raw_row_labels(mf)}')
print()

# ---- (b) chi^2 both ways -----------------------------------
print('PER-TRACER CHI^2 AT FIDUCIAL COSMOLOGY')
print('-' * 74)
print(f'{"Tracer":<12}{"zeff":>7}{"chi2 positional":>18}{"chi2 labelled":>16}{"ratio":>12}')
print('-' * 74)

tot_pos = tot_lab = 0.0
rows_out = []

for name, info in TRACERS.items():
    mf = Path(BAO_DIR) / f"{info['stem']}_mean.txt"
    cf = Path(BAO_DIR) / f"{info['stem']}_cov.txt"
    if not (mf.exists() and cf.exists()):
        continue

    mode = info['mode']
    keys = REQUIRED_KEYS[mode]
    icov = np.linalg.inv(load_cov(cf))

    lab_d = load_mean_labeled(mf)
    pos_d = load_mean_positional(mf, mode)

    theory, _ = get_bao_theory(FID['ombh2'], FID['omch2'], FID['H0'],
                               info['zeff'], mode)
    theory = np.asarray(theory, dtype=float)

    d_lab = np.array([lab_d[k] for k in keys]) - theory
    d_pos = np.array([pos_d[k] for k in keys]) - theory

    c_lab = float(d_lab @ icov @ d_lab)
    c_pos = float(d_pos @ icov @ d_pos)

    tot_lab += c_lab
    tot_pos += c_pos
    ratio = c_pos / c_lab if c_lab > 0 else np.nan
    rows_out.append((name, info['zeff'], c_pos, c_lab, ratio))
    print(f'{name:<12}{info["zeff"]:7.3f}{c_pos:18.4f}{c_lab:16.4f}{ratio:12.2f}')

print('-' * 74)
print(f'{"TOTAL":<12}{"":>7}{tot_pos:18.4f}{tot_lab:16.4f}'
      f'{tot_pos / tot_lab:12.2f}')
print()

# ---- (c) unambiguous statement of the inflation factor -----
lya = [r for r in rows_out if r[0] == 'Lya']
if lya:
    _, _, cp, cl, r = lya[0]
    print(f'Lya tracer alone : chi2 {cl:.4f} -> {cp:.4f}   (factor {r:.1f})')
print(f'DESI total       : chi2 {tot_lab:.4f} -> {tot_pos:.4f}   '
      f'(factor {tot_pos / tot_lab:.1f})')
print()
print('Quote whichever of these two you mean, and say which one it is.')

In [ ]:
from pathlib import Path

for stem in ['desi_2024_gaussian_bao_Lya_GCcomb',
             'desi_2024_gaussian_bao_LRG_GCcomb_z0.4-0.6']:
    hits = list(Path.home().rglob(f'{stem}_mean.txt'))
    print('=' * 60)
    print(stem, '->', len(hits), 'copy/copies found')
    for h in hits:
        print('  ', h)
    if hits:
        print('-' * 60)
        print(hits[0].read_text())

## Cell 5 — CAMB BAO Theory Engine

Computes DM/rd, DH/rd, DV/rd at each tracer redshift.

In [ ]:
def get_bao_theory(ombh2, omch2, H0, zeff, mode):
    """
    Compute BAO observables at zeff.
    Returns ([DM/rd, DH/rd], rd) for mode='aniso'
    Returns ([DV/rd],        rd) for mode='iso'
    """
    pars = camb.CAMBparams()
    pars.set_cosmology(H0=H0, ombh2=ombh2, omch2=omch2,
                       mnu=0.06, omk=0.0, tau=0.054)
    pars.set_dark_energy()
    pars.InitPower.set_params(As=2.1e-9, ns=0.965)
    pars.set_for_lmax(500, lens_potential_accuracy=0)
    results = camb.get_results(pars)

    rd  = results.get_derived_params()['rdrag']
    DA  = results.angular_diameter_distance(zeff)
    DM  = DA * (1.0 + zeff)
    H_z = results.hubble_parameter(zeff)
    DH  = 299792.458 / H_z
    DV  = (zeff * DM**2 * DH) ** (1.0 / 3.0)

    if mode == 'aniso':
        return np.array([DM / rd, DH / rd]), rd
    else:
        return np.array([DV / rd]), rd

# Sanity check against DESI DR1 Table 1
th, rd = get_bao_theory(0.0224, 0.120, 67.4, 0.510, 'aniso')
print(f'Sanity check LRG1 (z=0.510):')
print(f'  rd    = {rd:.3f} Mpc  (expect ~147)')
print(f'  DM/rd = {th[0]:.4f}   (DR1: 13.6200)')
print(f'  DH/rd = {th[1]:.4f}   (DR1: 20.9833)')

In [ ]:
# ===========================================================
#  RNAAS audit cell (v3) — fully self-contained.
#  Defines its own REQUIRED_KEYS and labelled loader, so it runs
#  in the OLD notebook that only has load_mean().
#  Requires: TRACERS, BAO_DIR, load_cov, get_bao_theory
# ===========================================================

import numpy as np
from pathlib import Path

FID = dict(ombh2=0.02239, omch2=0.11712, H0=67.41303)

REQUIRED_KEYS = {
    'aniso': ('DM_over_rs', 'DH_over_rs'),
    'iso':   ('DV_over_rs',),
}


def _rows(filepath):
    """(value, label) pairs in file order."""
    out = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            try:
                out.append((float(parts[1]), parts[2]))
            except ValueError:
                continue
    return out


def mean_labelled(filepath, mode):
    """Correct: assign by the label column."""
    d = {lab: val for val, lab in _rows(filepath)}
    return np.array([d[k] for k in REQUIRED_KEYS[mode]])


def mean_positional(filepath, mode):
    """Buggy: assign by row order, ignoring labels. This is load_mean()."""
    vals = [val for val, _ in _rows(filepath)]
    return np.array(vals[:len(REQUIRED_KEYS[mode])])


print(f'{"Tracer":<12}{"zeff":>7}{"chi2 positional":>18}{"chi2 labelled":>16}{"ratio":>12}')
print('-' * 65)

tot_pos = tot_lab = 0.0
rows_out = []

for name, info in TRACERS.items():
    mf = Path(BAO_DIR) / f"{info['stem']}_mean.txt"
    cf = Path(BAO_DIR) / f"{info['stem']}_cov.txt"
    if not (mf.exists() and cf.exists()):
        print(f'{name:<12} SKIPPED — files missing')
        continue

    mode = info['mode']
    try:
        icov = np.linalg.inv(load_cov(cf))
        theory, _ = get_bao_theory(FID['ombh2'], FID['omch2'], FID['H0'],
                                   info['zeff'], mode)
        theory = np.asarray(theory, dtype=float)

        d_lab = mean_labelled(mf, mode) - theory
        d_pos = mean_positional(mf, mode) - theory

        c_lab = float(d_lab @ icov @ d_lab)
        c_pos = float(d_pos @ icov @ d_pos)
    except Exception as e:
        print(f'{name:<12} FAILED — {type(e).__name__}: {e}')
        continue

    tot_lab += c_lab
    tot_pos += c_pos
    ratio = c_pos / c_lab if c_lab > 0 else float('nan')
    rows_out.append((name, info['zeff'], c_pos, c_lab, ratio))
    print(f'{name:<12}{info["zeff"]:7.3f}{c_pos:18.4f}{c_lab:16.4f}{ratio:12.2f}')

print('-' * 65)

if rows_out:
    tr = tot_pos / tot_lab if tot_lab > 0 else float('nan')
    print(f'{"TOTAL":<12}{"":>7}{tot_pos:18.4f}{tot_lab:16.4f}{tr:12.2f}')
    print()
    for r in rows_out:
        if r[0] == 'Lya':
            print(f'Lya alone  : {r[3]:.4f} -> {r[2]:.4f}   (factor {r[4]:.1f})')
    print(f'DESI total : {tot_lab:.4f} -> {tot_pos:.4f}   (factor {tr:.1f})')

## Cell 6 — Individual Log-Likelihoods

In [ ]:
def log_like_wmap9(theta):
    """
    WMAP9 CMB log-likelihood.
    Uses all 6 parameters: ombh2, omch2, H0, ns, ln10As, tau.
    """
    try:
        neg2, _ = wmap_like(*get_cls_camb_wmap(*theta))
        return -0.5 * neg2 if np.isfinite(neg2) else -np.inf
    except Exception:
        return -np.inf


def log_like_desi(ombh2, omch2, H0):
    """
    DESI DR1 BAO Gaussian likelihood.
    Only uses ombh2, omch2, H0. nS, As, tau do not affect BAO distances.
    No Planck rd prior — WMAP9 already calibrates rd via the CMB.
    """
    lnL = 0.0
    for name, d in bao_data.items():
        try:
            theory, _ = get_bao_theory(ombh2, omch2, H0, d['zeff'], d['mode'])
        except Exception:
            return -np.inf
        delta = d['mean'] - theory
        lnL  -= 0.5 * float(delta @ d['icov'] @ delta)
    return lnL if np.isfinite(lnL) else -np.inf


# Test both at WMAP9 best-fit
theta_wmap = np.array([0.02264, 0.1138, 70.0, 0.972, 3.089, 0.089])
ll_w = log_like_wmap9(theta_wmap)
ll_d = log_like_desi(theta_wmap[0], theta_wmap[1], theta_wmap[2])
print(f'WMAP9 lnL at WMAP9 BF : {ll_w:.2f}')
print(f'DESI  lnL at WMAP9 BF : {ll_d:.2f}')
print(f'Joint lnL              : {ll_w + ll_d:.2f}')

## Cell 7 — Joint Prior and Posterior

In [ ]:
def log_prior(theta):
    """Flat prior within bounds."""
    if np.all((theta >= PRIOR_BOUNDS[:, 0]) & (theta <= PRIOR_BOUNDS[:, 1])):
        return 0.0
    return -np.inf


def log_posterior_joint(theta):
    """
    Joint WMAP9 + DESI DR1 log-posterior.

    ln L_total = ln L_WMAP9(theta) + ln L_DESI_BAO(ombh2, omch2, H0)

    No Planck rd prior. WMAP9 provides the early-universe calibration
    of the sound horizon through the CMB acoustic peaks.
    DESI contributes only through ombh2, omch2, H0.
    """
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf

    # WMAP9 CMB — all 6 parameters
    ll_wmap = log_like_wmap9(theta)
    if not np.isfinite(ll_wmap):
        return -np.inf

    # DESI BAO — only ombh2, omch2, H0
    ll_desi = log_like_desi(theta[0], theta[1], theta[2])
    if not np.isfinite(ll_desi):
        return -np.inf

    return lp + ll_wmap + ll_desi


# Test joint posterior at WMAP9 best-fit
lp_test = log_posterior_joint(theta_wmap)
print(f'Joint log-posterior at WMAP9 BF: {lp_test:.2f}')
print(f'  = prior {log_prior(theta_wmap):.1f} + WMAP9 {ll_w:.2f} + DESI {ll_d:.2f}')

## Cell 8 — MAP Fit for Joint Posterior

In [ ]:
import time

def neg_log_posterior_joint(theta):
    val = log_posterior_joint(theta)
    return 1e10 if not np.isfinite(val) else -val

# Start from WMAP9 best-fit
theta0 = np.array([0.02264, 0.1138, 70.0, 0.972, 3.089, 0.089])

print('Single evaluation time test...')
t0 = time.time()
_ = neg_log_posterior_joint(theta0)
t_single = time.time() - t0
print(f'Single eval: {t_single:.2f} s')
print(f'Estimated MCMC: ~{t_single * NWALKERS * NSTEPS / 3600:.1f} hours')

print('\nRunning MAP optimisation...')
result = opt.minimize(
    neg_log_posterior_joint, theta0, method='Nelder-Mead',
    options=dict(xatol=1e-4, fatol=1e-2, maxiter=2000, disp=True)
)
theta_map = result.x

print(f'\n=== MAP — WMAP9 + DESI DR1 Joint ===')
print(f'{"Param":<12} {"MAP":>10} {"WMAP9-only":>12}')
print('-' * 38)
wmap9_vals = [0.02264, 0.1138, 70.0, 0.972, 3.089, 0.089]
for name, val, ref in zip(PARAM_NAMES, theta_map, wmap9_vals):
    print(f'{name:<12} {val:10.5f} {ref:12.5f}')

h_map   = theta_map[2] / 100.0
Omega_m = (theta_map[0] + theta_map[1]) / h_map**2
print(f'\nDerived:')
print(f'  Omega_m = {Omega_m:.4f}  (DESI DR1 paper: 0.295 +/- 0.015)')

## Cell 9 — MCMC Sampling

Joint likelihood. Each step requires one CAMB + WMAP9 Fortran call
plus one CAMB + BAO calculation. Estimated 8-16 hours on a single CPU.

In [ ]:
NDIM = 6
rng  = np.random.default_rng(42)
p0   = theta_map + 1e-3 * theta_map * rng.standard_normal((NWALKERS, NDIM))
p0   = np.clip(p0, PRIOR_BOUNDS[:, 0], PRIOR_BOUNDS[:, 1])

sampler = emcee.EnsembleSampler(
    NWALKERS, NDIM, log_posterior_joint,
    moves=emcee.moves.DEMove()
)

print(f'Joint MCMC: {NWALKERS} walkers x {NSTEPS} steps')
print(f'Likelihood : WMAP9 (8 components) + DESI DR1 BAO ({len(bao_data)} tracers)')
print(f'Parameters : {PARAM_NAMES}')
print()
sampler.run_mcmc(p0, NSTEPS, progress=True)
print('Done.')
print(f'Mean acceptance fraction: {np.mean(sampler.acceptance_fraction):.3f}')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
chain_file = CHAIN_DIR + f'/wmap9_desi_joint_chain_{timestamp}.npy'
flat_file  = CHAIN_DIR + f'/wmap9_desi_joint_flat_{timestamp}.npy'
np.save(chain_file, sampler.get_chain())
print(f'Full chain saved : {chain_file}')

## Cell 10 — Convergence Diagnostics

In [ ]:
try:
    tau_ac = sampler.get_autocorr_time(quiet=True)
    print('Autocorrelation time (primary diagnostic):')
    for name, t in zip(PARAM_NAMES, tau_ac):
        ratio  = NSTEPS / t
        status = 'CONVERGED' if ratio > 50 else ('OK' if ratio > 20 else 'NEED MORE STEPS')
        print(f'  tau({name:8s}) = {t:6.1f}   NSTEPS/tau = {ratio:5.1f}   [{status}]')
    NBURN_AUTO = max(NBURN, int(2 * np.max(tau_ac)))
except Exception as e:
    print(f'Autocorr: {e}')
    NBURN_AUTO = NBURN

print(f'\nBurn-in used: {NBURN_AUTO} steps')

chain     = sampler.get_chain()
post_burn = chain[NBURN_AUTO:]
n_groups  = 4
group_sz  = NWALKERS // n_groups
print('\nApproximate G-R R-hat (walker groups):')
for i, name in enumerate(PARAM_NAMES):
    gm   = [post_burn[:, j*group_sz:(j+1)*group_sz, i].mean() for j in range(n_groups)]
    gv   = [post_burn[:, j*group_sz:(j+1)*group_sz, i].var()  for j in range(n_groups)]
    W    = np.mean(gv)
    B    = np.var(gm) * len(post_burn)
    n    = len(post_burn)
    Rhat = np.sqrt(((n-1)/n * W + B/n) / W) if W > 0 else np.nan
    print(f'  R-hat({name:8s}) = {Rhat:.4f}  [{"OK" if Rhat < 1.05 else "check"}]')

## Cell 11 — Posterior Summary

In [ ]:
flat = sampler.get_chain(discard=NBURN_AUTO, flat=True)
np.save(flat_file, flat)
print(f'Flat samples shape : {flat.shape}')
print(f'Flat samples saved : {flat_file}')

# Reference values
wmap9_ref  = {'ombh2':(0.02264,0.00050),'omch2':(0.1138,0.0045),
               'H0':(69.32,0.80),'ns':(0.9710,0.0130),
               'ln10As':(3.089,0.025),'tau':(0.089,0.014)}
desi_ref   = {'ombh2':(None,None),'omch2':(None,None),
               'H0':(68.53,0.80),'ns':(None,None),
               'ln10As':(None,None),'tau':(None,None)}

print(f'\n=== WMAP9 + DESI DR1 Joint LCDM Posterior ===')
print(f'{"Param":<12} {"Median":>10} {"Std":>10} {"WMAP9-only":>12} {"Tension":>8}')
print('-' * 62)
for i, name in enumerate(PARAM_NAMES):
    s = flat[:, i]
    q16, q50, q84 = np.percentile(s, [16, 50, 84])
    rv, rs = wmap9_ref[name]
    t = abs(q50 - rv) / np.sqrt(s.std()**2 + rs**2)
    print(f'{name:<12} {q50:10.5f} {s.std():10.5f} {rv:12.5f} {t:7.2f}s')

print()
ombh2_s = flat[:, 0]
omch2_s = flat[:, 1]
H0_s    = flat[:, 2]
h_s     = H0_s / 100.0
Omega_m_s = (ombh2_s + omch2_s) / h_s**2
q16, q50, q84 = np.percentile(Omega_m_s, [16, 50, 84])
print(f'Derived Omega_m = {q50:.4f} +{q84-q50:.4f} -{q50-q16:.4f}')
print(f'DESI DR1 paper  : 0.295 +/- 0.015')

## Cell 12 — Trace Plot

In [ ]:
chain = sampler.get_chain()
fig, axes = plt.subplots(NDIM, 1, figsize=(12, 10), sharex=True)
for i, (ax, lbl) in enumerate(zip(axes, PARAM_LABELS)):
    ax.plot(chain[:, :, i], alpha=0.3, lw=0.5, color='steelblue')
    ax.axvline(NBURN_AUTO, color='red', ls='--', lw=1.5, label='burn-in')
    ax.set_ylabel(lbl, fontsize=9)
axes[0].legend(fontsize=8)
axes[-1].set_xlabel('Step')
fig.suptitle('MCMC Trace — WMAP9 + DESI DR1 Joint LCDM', fontsize=12)
plt.tight_layout()
fname = IMG_DIR + '/wmap9_desi_joint_trace.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fname}')

## Cell 13 — Corner Plot

In [ ]:
fig = corner.corner(
    flat,
    labels=PARAM_LABELS,
    quantiles=[0.16, 0.50, 0.84],
    show_titles=True,
    title_kwargs={'fontsize': 9},
    smooth=1.0, smooth1d=1.0,
    truths=[0.02264, 0.1138, 69.32, 0.9710, 3.089, 0.089],
    truth_color='steelblue'
)
fig.suptitle('WMAP9 + DESI DR1 Joint \u039bCDM Posterior', y=1.01, fontsize=12)
fname = IMG_DIR + '/wmap9_desi_joint_corner.png'
fig.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fname}')

## Cell 14 — 1D Comparison: WMAP9 vs DESI vs Joint

In [ ]:
# Load existing individual chains
wmap_file = PY_DIR + '/wmap9_official_lcdm_mcmc_chains.npy'
desi_file = CHAIN_DIR + '/desi_bao_lcdm_flat_20260615_184435.npy'

datasets = {'WMAP9 + DESI Joint': flat}
colours  = {
    'WMAP9 + DESI Joint': 'green',
    'WMAP9 alone'        : 'steelblue',
    'DESI DR1 alone'     : 'darkorange',
}

if os.path.exists(wmap_file):
    wc = np.load(wmap_file)
    datasets['WMAP9 alone'] = wc
    print(f'Loaded WMAP9 alone  : {wc.shape[0]} samples')
else:
    print(f'WMAP9 chain not found: {wmap_file}')

if os.path.exists(desi_file):
    dc = np.load(desi_file)
    if dc.ndim == 3:
        dc = dc[500:].reshape(-1, dc.shape[-1])
    datasets['DESI DR1 alone'] = dc
    print(f'Loaded DESI DR1     : {dc.shape[0]} samples')
else:
    print(f'DESI chain not found: {desi_file}')

# Plot shared parameters: ombh2, omch2, H0
shared_idx   = [0, 1, 2]
shared_names = [PARAM_LABELS[i] for i in shared_idx]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for j, (ax, lbl, idx) in enumerate(zip(axes, shared_names, shared_idx)):
    for dname, samples in datasets.items():
        col = samples[:, idx] if samples.ndim == 2 else samples[:, idx]
        ax.hist(col, bins=80, density=True,
                alpha=0.5, color=colours[dname], label=dname)
        ax.axvline(np.median(col), color=colours[dname], ls='--', lw=1.5)
    ax.set_xlabel(lbl, fontsize=11)
    ax.set_ylabel('P (normalised)', fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle('LCDM: WMAP9 alone vs DESI DR1 alone vs Joint',
             fontsize=13, y=1.02)
plt.tight_layout()
fname = IMG_DIR + '/wmap9_desi_joint_comparison.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fname}')

## Cell 15 — Parameter Tension Summary

In [ ]:
print('='*65)
print('  WMAP9 + DESI DR1 Joint LCDM — Final Summary')
print('='*65)
print(f'  Likelihood : WMAP9 official (8 components) + DESI DR1 BAO')
print(f'  Tracers    : {len(bao_data)} (BGS, LRG1, LRG2, LRG3+ELG1, ELG2, QSO, Lya)')
print(f'  Walkers    : {NWALKERS}')
print(f'  Steps      : {NSTEPS}  (burn-in: {NBURN_AUTO})')
print(f'  Flat samples: {flat.shape[0]}')
print()

# Individual dataset medians for tension calculation
medians = {}
stds    = {}
for i, name in enumerate(PARAM_NAMES):
    medians[name] = np.median(flat[:, i])
    stds[name]    = flat[:, i].std()

wmap9_medians = {'ombh2': 0.02261, 'omch2': 0.1141,
                 'H0': 69.35, 'ns': 0.972, 'ln10As': 3.089, 'tau': 0.089}
desi_medians  = {'ombh2': 0.02281, 'omch2': 0.11822, 'H0': 69.04}

print(f'  {"Param":<12} {"Joint":>10} {"WMAP9":>10} {"DESI":>10} {"Shift":>8}')
print('  ' + '-'*56)
for name in PARAM_NAMES:
    jval = medians[name]
    wval = wmap9_medians.get(name, None)
    dval = desi_medians.get(name, None)
    shift = abs(jval - wval) / stds[name] if wval else 0
    dstr  = f'{dval:10.5f}' if dval else '         -'
    print(f'  {name:<12} {jval:10.5f} {wval:10.5f} {dstr} {shift:6.2f}s')

print()
h_j    = medians['H0'] / 100.0
Omm_j  = (medians['ombh2'] + medians['omch2']) / h_j**2
print(f'  Derived Omega_m (joint) = {Omm_j:.4f}')
print(f'  DESI DR1 paper           = 0.295 +/- 0.015')
print()
print('  The joint H0 sits between WMAP9 and DESI, with tighter')
print('  uncertainty because the datasets are complementary.')
print('='*65)